# Backfill Tuner Columns — Orchestrator

Runs [Backfill Tuner Columns Daily](/editor/notebooks/2955101215936593) once per day across the last 2 years using a controlled pool of **3–5 concurrent notebook runs**.

## How it works
1. Generates a list of dates (default: 2 years ago → yesterday)
2. Skips dates already fully populated in `viewing_content_golden`
3. Dispatches each date to the daily notebook via `dbutils.notebook.run` in parallel
4. Reports success/failure per date at the end

## Job setup
Schedule this orchestrator notebook as a single-task Databricks job.
- All parallelism is managed internally — job concurrency can stay at **1**
- Pass `start_date` / `end_date` as job parameters to reprocess a custom range
- Adjust `concurrency` (default 5) based on cluster size

In [0]:
from datetime import datetime, timedelta

# Optional overrides — leave blank to use defaults
dbutils.widgets.text("start_date",       "",  "Start Date (YYYY-MM-DD, blank = 2 years ago)")
dbutils.widgets.text("end_date",         "",  "End Date   (YYYY-MM-DD, blank = yesterday)")
dbutils.widgets.dropdown("concurrency",  "5", ["1", "2", "3", "4", "5", "10"], "Concurrent runs")
dbutils.widgets.text("timeout_minutes",  "60", "Timeout per run (minutes)")

_start = dbutils.widgets.get("start_date")
_end   = dbutils.widgets.get("end_date")
concurrency     = int(dbutils.widgets.get("concurrency"))
timeout_seconds = int(dbutils.widgets.get("timeout_minutes")) * 60

today      = datetime.now().date()
end_date   = datetime.strptime(_end,   '%Y-%m-%d').date() if _end   else today - timedelta(days=1)
start_date = datetime.strptime(_start, '%Y-%m-%d').date() if _start else end_date - timedelta(days=730)

print(f"Date range  : {start_date}  to  {end_date}")
print(f"Concurrency : {concurrency} parallel runs")
print(f"Timeout     : {timeout_seconds // 60} min per run")

In [0]:
# Generate full candidate list
all_dates = []
current = start_date
while current <= end_date:
    all_dates.append(current.strftime('%Y-%m-%d'))
    current += timedelta(days=1)

print(f"Total candidate dates : {len(all_dates)}")

# Skip dates where all three columns are already filled for that day
# A date is 'done' when no row for that day has all three columns NULL
already_done_df = spark.sql("""
    SELECT DISTINCT CAST(session_start_hour AS DATE) AS done_date
    FROM prod.detection.viewing_content_golden
    WHERE session_start_hour >= '{start}'
      AND session_start_hour <  '{end}'
      AND tuner_series_id    IS NOT NULL
      AND tuner_tms_epi_title  IS NOT NULL
      AND tuner_tivo_epi_title IS NOT NULL
""".format(start=start_date, end=end_date + timedelta(days=1)))

done_dates = {row['done_date'].strftime('%Y-%m-%d') for row in already_done_df.collect()}
date_list  = [d for d in all_dates if d not in done_dates]

print(f"Already populated     : {len(done_dates)}")
print(f"Dates to process      : {len(date_list)}")
if date_list:
    print(f"First: {date_list[0]}   Last: {date_list[-1]}")

In [0]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

BACKFILL_NOTEBOOK = "/Users/mohit.gangwani@vizio.com/Backfill Tuner Columns Daily"

results   = {"success": [], "failed": []}

def run_one_day(date_str):
    """Run the daily backfill notebook for a single date."""
    return dbutils.notebook.run(
        BACKFILL_NOTEBOOK,
        timeout_seconds=timeout_seconds,
        arguments={"target_date": date_str}
    )

if not date_list:
    print("Nothing to process — all dates already populated.")
else:
    print(f"Starting {len(date_list)} runs at concurrency={concurrency} ...\n")
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        futures = {executor.submit(run_one_day, d): d for d in date_list}
        for future in as_completed(futures):
            date = futures[future]
            try:
                future.result()
                results["success"].append(date)
                print(f"{date}  ({len(results['success']) + len(results['failed'])}/{len(date_list)})")
            except Exception as exc:
                results["failed"].append({"date": date, "error": str(exc)})
                print(f"{date}  FAILED: {exc}")

    elapsed = time.time() - t0
    print(f"\nFinished in {elapsed / 60:.1f} min  |  "
          f"Success: {len(results['success'])}  |  "
          f"Failed: {len(results['failed'])}")

In [0]:
# Print any failures so the job log captures them clearly
if results["failed"]:
    print(f"\n{'='*50}")
    print(f"FAILED DATES ({len(results['failed'])}) — re-run with start_date / end_date overrides:")
    print(f"{'='*50}")
    for entry in sorted(results["failed"], key=lambda x: x["date"]):
        print(f"  {entry['date']}  →  {entry['error']}")
    # Surface failures to the Databricks job as a notebook exit value
    dbutils.notebook.exit(f"FAILED: {[e['date'] for e in results['failed']]}")
else:
    print("All dates processed successfully.")
    dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
SELECT DATE(session_start)
, tuner_series_id IS NOT NULL AS tuner_series_id
, tuner_tms_epi_title IS NOT NULL AS tuner_tms_epi_title
, tuner_tivo_epi_title IS NOT NULL AS tuner_tivo_epi_title
, COUNT(*)
FROM prod.detection.viewing_content_golden
WHERE COALESCE(tuner_tms_title, tuner_tivo_title, tuner_tms_episode_id, tuner_tivo_episode_id) IS NOT NULL
AND session_start >= '2026-05-28'
AND session_start < '2026-06-25'
GROUP BY 1, 2, 3, 4